# 05b - IEMap Atlas Registration

This notebook is an alternate Step 5 workflow for registering the prepared histology stack to the IEMap atlas. The first goal is a working T1 atlas-registration dry run, then a live baseline run, and only then quality tuning.

By default this notebook performs preflight checks and stays in dry-run/planning mode until an initialization is explicitly supplied with `INIT_AFFINE_PATH` or `ORIENTATION_FROM` plus `ORIENTATION_TO`. CT and microCT IEMap volumes are left for later because they are on different grids from `segIEMap.nii`.

## Step 1: Setup and Parameters

In [ ]:
from __future__ import annotations

import json
import shlex
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import nibabel as nib
except ImportError as exc:
    raise ImportError("Install nibabel to run IEMap NIfTI preflight checks.") from exc

from IPython.display import Image, JSON, display

from wsi_pipeline.emlddmm_prep import ensure_emlddmm_on_path
from wsi_pipeline.registration import plan_emlddmm_workflow, run_emlddmm_workflow
from wsi_pipeline.registration.backend import resolve_emlddmm_backend
from wsi_pipeline.registration.orientation import describe_orientation_transform, validate_orientation_code
from wsi_pipeline.registration.targets import load_manifest

EMLDDMM_REPO = ensure_emlddmm_on_path("emlddmm")

In [ ]:
# Prepared histology target from notebook 04.
DATASET_ROOT = Path("/cis/home/dpadova/Documents/temporal_bone_project/notebook_flatfile_smoke_level7/tissue_sections").expanduser()
TARGET_SOURCE = DATASET_ROOT
TARGET_SOURCE_FORMAT = "auto"  # auto, prepared-dir, or precomputed
PRECOMPUTED_MANIFEST = None
REGISTRATION_OUTPUT = DATASET_ROOT / "emlddmm_iemap"
PRESET = "macaque-notebook"
DEVICE = "auto"

# Keep the default conservative until atlas initialization is set and preflight looks sane.
RUN_REGISTRATION = False
DRY_RUN = True
WRITE_QC_REPORT = True
WRITE_NOTEBOOK_BUNDLE = False

# IEMap atlas resources. The segmentation label map is expected to align with the T1 grid.
IEMAP_ATLAS_ROOT = Path("/cis/home/dpadova/Documents/git/iemap-atlas").expanduser()
IEMAP_ATLAS_MODALITY = "T1"
IEMAP_ATLAS_FILES = {
    "CISS": "IEMap_CISS.nii.gz",
    "CT": "IEMap_CT.nii.gz",
    "microCT": "IEMap_microCT.nii.gz",
    "T1": "IEMap_T1.nii.gz",
    "T2": "IEMap_T2.nii.gz",
}
ATLAS_PATH = IEMAP_ATLAS_ROOT / IEMAP_ATLAS_FILES[IEMAP_ATLAS_MODALITY]
LABEL_PATH = IEMAP_ATLAS_ROOT / "segIEMap.nii"

# Atlas initialization. Set one of these before running the workflow plan.
INIT_AFFINE_PATH = None
ORIENTATION_FROM = None  # Example: "PIR"
ORIENTATION_TO = None  # Example: "RIP"
ATLAS_UNIT_SCALE = None  # Preset default: 1000.0
TARGET_UNIT_SCALE = None  # Preset default: 1.0

# Atlas registration config. The workflow already passes the prepared target tissue mask as W0.
USE_TARGET_MASK = True
SIGMA_M = None
SIGMA_A = None
SIGMA_B = None
PRIORS = None
UPDATE_PRIORS = True
N_E_STEP = 5

# Optional atlas-domain preprocessing. Leave disabled until baseline IEMap QC has been inspected.
ENABLE_ATLAS_ROI_PREPROCESSING = False
ATLAS_ROI_THRESHOLD = 0.0
ATLAS_ROI_FILL_VALUE = 0.0

# Step 5 atlas registration does not perform between-slice upsampling.
UPSAMPLE_BETWEEN_SLICES = False
UPSAMPLE_MODE = "img"
RUN_TRANSFORMATION_GRAPH = False
TRANSFORMATION_GRAPH_SCRIPT = None

# Optional base EM-LDDMM config. This notebook writes an effective config under REGISTRATION_OUTPUT/inputs.
EMLDDMM_CONFIG = None

In [ ]:
def optional_path(value):
    if value in (None, ""):
        return None
    return Path(value).expanduser()


def write_json(path: Path, payload: dict) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True))
    return path


def add_if_not_none(payload: dict, key: str, value):
    if value is not None:
        payload[key] = value


atlas_path = optional_path(ATLAS_PATH)
label_path = optional_path(LABEL_PATH)
init_affine_path = optional_path(INIT_AFFINE_PATH)
transformation_graph_script = optional_path(TRANSFORMATION_GRAPH_SCRIPT)
base_config_path = optional_path(EMLDDMM_CONFIG)
inputs_dir = REGISTRATION_OUTPUT / "inputs"
inputs_dir.mkdir(parents=True, exist_ok=True)

if not USE_TARGET_MASK:
    raise ValueError("The workflow currently passes the prepared target tissue mask as W0; leave USE_TARGET_MASK=True.")

if IEMAP_ATLAS_MODALITY not in IEMAP_ATLAS_FILES:
    raise ValueError(f"Unknown IEMap modality: {IEMAP_ATLAS_MODALITY!r}")

if atlas_path is None:
    raise ValueError("ATLAS_PATH must resolve to an IEMap NIfTI volume.")

if ENABLE_ATLAS_ROI_PREPROCESSING:
    atlas_img = nib.load(str(atlas_path))
    atlas_data = np.asanyarray(atlas_img.dataobj).astype(np.float32)
    atlas_mask = atlas_data > float(ATLAS_ROI_THRESHOLD)
    masked_data = np.where(atlas_mask, atlas_data, float(ATLAS_ROI_FILL_VALUE)).astype(np.float32)
    masked_atlas_path = inputs_dir / f"iemap_{IEMAP_ATLAS_MODALITY.lower()}_roi_masked.nii.gz"
    nib.save(nib.Nifti1Image(masked_data, atlas_img.affine, atlas_img.header), str(masked_atlas_path))
    atlas_path = masked_atlas_path

atlas_registration_extra_kwargs = {}
add_if_not_none(atlas_registration_extra_kwargs, "sigmaM", SIGMA_M)
add_if_not_none(atlas_registration_extra_kwargs, "sigmaA", SIGMA_A)
add_if_not_none(atlas_registration_extra_kwargs, "sigmaB", SIGMA_B)
add_if_not_none(atlas_registration_extra_kwargs, "priors", PRIORS)
add_if_not_none(atlas_registration_extra_kwargs, "update_priors", UPDATE_PRIORS)
add_if_not_none(atlas_registration_extra_kwargs, "n_e_step", N_E_STEP)

config_payload = {}
if base_config_path is not None:
    config_payload = json.loads(base_config_path.read_text())
config_payload.setdefault("atlas_registration", {}).setdefault("extra_kwargs", {}).update(atlas_registration_extra_kwargs)

effective_config_path = write_json(inputs_dir / "iemap_atlas_registration_config.json", config_payload)

orientation_from = ORIENTATION_FROM.upper() if ORIENTATION_FROM else None
orientation_to = ORIENTATION_TO.upper() if ORIENTATION_TO else None
if orientation_from:
    validate_orientation_code(orientation_from)
if orientation_to:
    validate_orientation_code(orientation_to)
initialization_ready = bool(init_affine_path) or bool(orientation_from and orientation_to)

setup_payload = {
    "atlas_path": str(atlas_path),
    "label_path": str(label_path) if label_path else None,
    "effective_config_path": str(effective_config_path),
    "target_mask_W0_enabled": USE_TARGET_MASK,
    "atlas_registration_extra_kwargs": atlas_registration_extra_kwargs,
    "atlas_roi_preprocessing_enabled": ENABLE_ATLAS_ROI_PREPROCESSING,
    "initialization_ready": initialization_ready,
}
display(JSON(setup_payload, expanded=True))

if orientation_from and orientation_to:
    print(describe_orientation_transform(orientation_from, orientation_to))
elif not initialization_ready:
    print("Set INIT_AFFINE_PATH or both ORIENTATION_FROM and ORIENTATION_TO before resolving the atlas-registration plan.")

## Step 2: Validate Prepared Target Handoff

In [ ]:
samples_path = DATASET_ROOT / "samples.tsv"
manifest_path = DATASET_ROOT / "precomputed" / "manifest.json"
sidecar_dir = DATASET_ROOT / "sidecars"

handoff = {
    "dataset_root": str(DATASET_ROOT),
    "samples_tsv": samples_path.exists(),
    "precomputed_manifest": manifest_path.exists(),
    "sidecar_dir": sidecar_dir.exists(),
    "target_source": str(TARGET_SOURCE),
    "target_source_format": TARGET_SOURCE_FORMAT,
}
display(JSON(handoff, expanded=True))

missing = [name for name, ok in handoff.items() if name.endswith(("tsv", "manifest", "dir")) and not ok]
if missing:
    raise FileNotFoundError(f"Missing Step 4 handoff files: {missing}")

manifest = load_manifest(manifest_path)
entries = manifest.get("entries") or manifest.get("slices") or []
print(f"Manifest entries: {len(entries)}")

## Step 3: IEMap Atlas Preflight

In [ ]:
def nifti_summary(path: Path, *, label: bool = False) -> dict:
    if path is None:
        return {"path": None, "exists": False}
    info = {"path": str(path), "exists": path.exists()}
    if not path.exists():
        return info
    img = nib.load(str(path))
    hdr = img.header
    info.update(
        {
            "shape": list(img.shape),
            "dtype": str(hdr.get_data_dtype()),
            "zooms": [float(v) for v in hdr.get_zooms()[: len(img.shape)]],
            "xyzt_units": tuple(str(v) for v in hdr.get_xyzt_units()),
            "affine": np.asarray(img.affine).round(6).tolist(),
        }
    )
    data = np.asanyarray(img.dataobj)
    finite = data[np.isfinite(data)]
    if finite.size:
        info["finite_min"] = float(finite.min())
        info["finite_max"] = float(finite.max())
    if label and finite.size:
        unique = np.unique(finite)
        info["label_count"] = int(unique.size)
        info["label_preview"] = [int(v) if float(v).is_integer() else float(v) for v in unique[:20]]
    return info

atlas_info = nifti_summary(atlas_path)
label_info = nifti_summary(label_path, label=True) if label_path else {"path": None, "exists": False}

atlas_img = nib.load(str(atlas_path)) if atlas_path and atlas_path.exists() else None
label_img = nib.load(str(label_path)) if label_path and label_path.exists() else None
label_grid_compatible = None
if atlas_img is not None and label_img is not None:
    label_grid_compatible = {
        "same_shape": atlas_img.shape[:3] == label_img.shape[:3],
        "same_zooms": np.allclose(atlas_img.header.get_zooms()[:3], label_img.header.get_zooms()[:3]),
        "same_affine": np.allclose(atlas_img.affine, label_img.affine),
    }

preflight_payload = {
    "requested_modality": IEMAP_ATLAS_MODALITY,
    "atlas": atlas_info,
    "label": label_info,
    "label_grid_compatible": label_grid_compatible,
}
display(JSON(preflight_payload, expanded=False))

if IEMAP_ATLAS_MODALITY in {"CT", "microCT"}:
    print("CT and microCT are intentionally deferred: segIEMap.nii is expected to match the T1 grid, not these grids.")

if label_grid_compatible and not all(label_grid_compatible.values()):
    print("Label-grid mismatch detected. Use T1 + segIEMap.nii for the first atlas-registration pass.")

## Step 4: Target Mask Coverage Preflight

In [ ]:
def is_present(entry: dict) -> bool:
    if "present" in entry:
        return bool(entry["present"])
    if "status" in entry:
        return str(entry["status"]).lower() in {"present", "available", "ok"}
    return not bool(entry.get("missing", False))

present_entries = [entry for entry in entries if is_present(entry)]
grid_indices = sorted(
    int(entry.get("grid_index", idx))
    for idx, entry in enumerate(entries)
    if is_present(entry) and entry.get("grid_index") is not None
)
gaps = [b - a for a, b in zip(grid_indices[:-1], grid_indices[1:])]
manifest_mask_payload = {
    "present_manifest_entries": len(present_entries),
    "total_manifest_entries": len(entries),
    "present_fraction": len(present_entries) / len(entries) if entries else None,
    "max_present_slice_gap": max(gaps) if gaps else 0,
}

backend = resolve_emlddmm_backend()
target_mask_payload = {"computed": False}
try:
    xJ, J, title, names = backend.read_data(str(TARGET_SOURCE))
    names = [str(name) for name in names]
    mask_index = names.index("mask") if "mask" in names else len(names) - 1
    W0 = np.asarray(J[mask_index]) > 0
    per_slice_fraction = W0.reshape(W0.shape[0], -1).mean(axis=1)
    target_mask_payload = {
        "computed": True,
        "image_channels": names,
        "mask_channel": names[mask_index],
        "mask_shape": list(W0.shape),
        "mask_voxel_fraction": float(W0.mean()),
        "slices_with_mask": int(np.count_nonzero(per_slice_fraction > 0)),
        "max_slice_fraction": float(per_slice_fraction.max()) if per_slice_fraction.size else None,
        "median_nonzero_slice_fraction": float(np.median(per_slice_fraction[per_slice_fraction > 0]))
        if np.any(per_slice_fraction > 0)
        else 0.0,
    }
except Exception as exc:
    target_mask_payload = {"computed": False, "error": repr(exc)}

display(JSON({"manifest": manifest_mask_payload, "target_mask": target_mask_payload}, expanded=True))

## Step 5: Resolve Atlas-Registration Plan

In [ ]:
workflow_kwargs = dict(
    dataset_root=DATASET_ROOT,
    output_dir=REGISTRATION_OUTPUT,
    target_source=TARGET_SOURCE,
    target_source_format=TARGET_SOURCE_FORMAT,
    registration_output=REGISTRATION_OUTPUT,
    atlas=atlas_path,
    label=label_path,
    emlddmm_config=effective_config_path,
    preset=PRESET,
    init_affine=init_affine_path,
    orientation_from=orientation_from,
    orientation_to=orientation_to,
    atlas_unit_scale=ATLAS_UNIT_SCALE,
    target_unit_scale=TARGET_UNIT_SCALE,
    device=DEVICE,
    precomputed_manifest=optional_path(PRECOMPUTED_MANIFEST),
    upsample_between_slices=UPSAMPLE_BETWEEN_SLICES,
    upsample_mode=UPSAMPLE_MODE,
    run_transformation_graph=RUN_TRANSFORMATION_GRAPH,
    transformation_graph_script=transformation_graph_script,
    write_notebook_bundle=WRITE_NOTEBOOK_BUNDLE,
    write_qc_report=WRITE_QC_REPORT,
)

if not initialization_ready:
    resolved_plan = None
    print("Atlas registration needs initialization before the workflow plan can be resolved.")
    print("Set INIT_AFFINE_PATH or ORIENTATION_FROM/ORIENTATION_TO, then rerun from Step 1.")
else:
    resolved_plan = plan_emlddmm_workflow(**workflow_kwargs)
    plan_overview = {
        "mode": resolved_plan.mode,
        "backend": resolved_plan.backend_name,
        "enabled_stages": resolved_plan.enabled_stages,
        "target_downsampling": resolved_plan.target_downsampling,
        "atlas_downsampling": resolved_plan.atlas_downsampling,
        "target_working_spacing_um": resolved_plan.resolution_policy.get("target_working_spacing_um"),
        "atlas_reference_spacing_um": resolved_plan.resolution_policy.get("atlas_reference_spacing_um"),
        "atlas_path": str(atlas_path),
        "label_path": str(label_path) if label_path else None,
    }
    display(JSON(plan_overview, expanded=True))
    display(JSON(resolved_plan.resolution_policy, expanded=True))
    if resolved_plan.warnings:
        print("Warnings:")
        for warning in resolved_plan.warnings:
            print(f"- {warning}")

## Step 6: CLI Replay

In [ ]:
cmd = [
    "python",
    "scripts/run_pipeline.py",
    "step5",
    "--dataset-root",
    str(DATASET_ROOT),
    "--target-source",
    str(TARGET_SOURCE),
    "--target-source-format",
    TARGET_SOURCE_FORMAT,
    "--registration-output",
    str(REGISTRATION_OUTPUT),
    "--preset",
    PRESET,
    "--device",
    DEVICE,
    "--atlas",
    str(atlas_path),
    "--emlddmm-config",
    str(effective_config_path),
]
if label_path:
    cmd += ["--label", str(label_path)]
if init_affine_path:
    cmd += ["--init-affine", str(init_affine_path)]
if orientation_from:
    cmd += ["--orientation-from", orientation_from]
if orientation_to:
    cmd += ["--orientation-to", orientation_to]
if DRY_RUN:
    cmd.append("--dry-run")
if WRITE_QC_REPORT:
    cmd.append("--write-qc-report")

print(" \\\n  ".join(shlex.quote(part) for part in cmd))
if not initialization_ready:
    print("\nAdd --init-affine or --orientation-from/--orientation-to before running the command.")

## Step 7: Run Atlas Registration

In [ ]:
if RUN_REGISTRATION and resolved_plan is not None:
    result = run_emlddmm_workflow(**workflow_kwargs, dry_run=DRY_RUN)
    print(f"Mode:         {result.mode}")
    print(f"Plan JSON:    {result.plan_path}")
    print(f"Summary JSON: {result.summary_path}")
    print("Executed stages:", ", ".join(result.executed_stages) or "none")
elif RUN_REGISTRATION:
    result = None
    print("RUN_REGISTRATION=True, but atlas initialization is missing. Resolve the plan first.")
else:
    result = None
    print("RUN_REGISTRATION=False. Set it to True after preflight and initialization are ready.")

## Step 8: Review Atlas Outputs

In [ ]:
summary_path = REGISTRATION_OUTPUT / "workflow_summary.json"
if summary_path.exists():
    display(JSON(json.loads(summary_path.read_text()), expanded=False))
else:
    print(f"No workflow summary found yet: {summary_path}")

atlas_stage_dir = REGISTRATION_OUTPUT / "atlas_registration"
qc_candidates = [
    atlas_stage_dir / "atlas_registration_overview.png",
    atlas_stage_dir / "transformed_label_overview.png",
    atlas_stage_dir / "WM_overview.png",
    atlas_stage_dir / "WA_overview.png",
    atlas_stage_dir / "WB_overview.png",
]
for path in qc_candidates:
    if path.exists():
        print(path.name)
        display(Image(filename=str(path)))

report_path = REGISTRATION_OUTPUT / "qc_report.md"
if report_path.exists():
    print(report_path)

## Masking and Weighting Notes

The prepared target tissue mask is passed by the pipeline as `W0` into the atlas-registration backend call. In modern `emlddmm`, the target-domain matching cost uses the atlas-side mask/weights with the target mask (`WM * W0`), so missing target field of view is treated as a first-class registration concern.

The EM weighting controls above are passed through `atlas_registration.extra_kwargs`: `sigmaM`, `sigmaA`, `sigmaB`, `priors`, `update_priors`, and `n_e_step`. Leave atlas ROI preprocessing disabled for the first IEMap T1 baseline run, then tune a derived masked atlas only after reviewing transformed labels, atlas QC overlays, target mask coverage, and the `WM/WA/WB` outputs.

Local implementation references: `src/wsi_pipeline/registration/workflow.py` passes `W0` to `emlddmm_multiscale`, and `/cis/home/dpadova/Documents/git/emlddmm/emlddmm.py` applies the target mask in the weighted matching path. Reference examples from the older `torch-lddmm` API remain useful conceptually for matching-cost masking and voxelwise weight estimation: https://github.com/brianlee324/torch-lddmm

## Output Layout

The IEMap registration directory contains:

- `plan.json` - resolved atlas-registration plan and inputs.
- `workflow_summary.json` - executed or dry-run summary.
- `inputs/iemap_atlas_registration_config.json` - effective masking and EM weighting overrides from this notebook.
- `atlas_registration/` - atlas-registration stage inputs, transformed atlas products, labels, and QC artifacts.
- `qc_report.md` - optional QC report.

Between-slice upsampling stays in `05_emlddmm_registration.ipynb` and should be run only after alignment QC looks satisfactory.